# Build Skopje hourly online features with missing PM as NaN

This notebook rebuilds the online streaming CSV from the full weather grid and raw Pulse Eco measurements.

Goal:
- keep one hourly row per sensor/timestamp from the weather grid
- attach observed PM10/PM2.5 where Pulse Eco has data
- leave missing PM10/PM2.5 as blank CSV cells, which pandas reads back as NaN
- avoid interpolation, ffill, bfill, or median filling

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "feature_engineering":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_WEATHER_CSV = PROJECT_ROOT / "data" / "streaming" / "skopje_sensor_weather_features_online.csv"
PULSE_DIR = PROJECT_ROOT.parent / "common" / "notebooks" / "pulse_data"

# Keep the same sensor-quality rule as the observed-only CSV.
# Set to None if you want to keep every sensor, including sensors with 100% missing PM values.
MAX_MISSING_PERCENT = 50.0

# Optional: fill only short PM gaps after joining raw Pulse measurements.
# Long outages stay NaN so we do not fabricate weeks of pollution values.
INTERPOLATE_SHORT_GAPS = True
MAX_INTERPOLATION_GAP_HOURS = 6

output_name = (
    "skopje_sensor_weather_features_online_short_gap_interpolated.csv"
    if INTERPOLATE_SHORT_GAPS
    else "skopje_sensor_weather_features_online_hourly_nan.csv"
)
OUTPUT_CSV = PROJECT_ROOT / "data" / "streaming" / output_name

SOURCE_WEATHER_CSV, PULSE_DIR, OUTPUT_CSV

(PosixPath('/home/nikola/realtime-air-quality-forecasting/skopje/data/streaming/skopje_sensor_weather_features_online.csv'),
 PosixPath('/home/nikola/realtime-air-quality-forecasting/common/notebooks/pulse_data'),
 PosixPath('/home/nikola/realtime-air-quality-forecasting/skopje/data/streaming/skopje_sensor_weather_features_online_short_gap_interpolated.csv'))

## 1. Load the weather grid

The original online CSV already has the hourly weather rows we need. We drop the old PM columns because those were filled/imputed before. Then we join raw Pulse PM values again.

In [3]:
weather = pd.read_csv(SOURCE_WEATHER_CSV)
weather = weather.drop(columns=["pm10", "pm25"], errors="ignore")

weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True, errors="coerce").dt.floor("h")
weather["sensorId"] = weather["sensorId"].astype(str)

weather = (
    weather
    .dropna(subset=["timestamp", "sensorId"])
    .sort_values(["sensorId", "timestamp"])
    .drop_duplicates(subset=["sensorId", "timestamp"], keep="last")
    .reset_index(drop=True)
)

print(f"Weather grid rows: {len(weather):,}")
print(f"Sensors in weather grid: {weather['sensorId'].nunique():,}")
print(f"Date range: {weather['timestamp'].min()} -> {weather['timestamp'].max()}")
weather.head()

Weather grid rows: 406,224
Sensors in weather grid: 186
Date range: 2025-11-30 22:00:00+00:00 -> 2026-03-01 21:00:00+00:00


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure
0,2025-11-30 22:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,2.50,94.795290,2.817445,26.564985,947.18340
1,2025-11-30 23:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.00,95.766060,2.340000,22.619910,947.08930
2,2025-12-01 00:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.30,89.411446,2.500640,30.256361,947.44305
3,2025-12-01 01:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,0.75,89.692825,1.938659,21.801476,947.67750
4,2025-12-01 02:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,0.50,89.672714,0.742159,14.036275,947.42890


## 2. Load raw Pulse PM measurements

Pulse timestamps are converted to UTC and floored to the hour so they align with the weather grid. If multiple readings land in the same sensor/hour/type, we use the mean.

In [4]:
pulse_files = sorted(PULSE_DIR.rglob("*.csv"))
print(f"Pulse files found: {len(pulse_files):,}")

pulse_parts = []
for file_path in pulse_files:
    part = pd.read_csv(file_path)
    part["source_file"] = str(file_path.relative_to(PROJECT_ROOT.parent))
    pulse_parts.append(part)

pulse = pd.concat(pulse_parts, ignore_index=True)
pulse = pulse[pulse["type"].isin(["pm10", "pm25"])].copy()

pulse["timestamp"] = pd.to_datetime(pulse["timestamp"], utc=True, errors="coerce").dt.floor("h")
pulse["sensorId"] = pulse["sensorId"].astype(str)
pulse["value"] = pd.to_numeric(pulse["value"], errors="coerce")

pulse = pulse.dropna(subset=["timestamp", "sensorId", "type", "value"])

hourly_pm = (
    pulse
    .groupby(["sensorId", "timestamp", "type"], as_index=False)["value"]
    .mean()
    .pivot(index=["sensorId", "timestamp"], columns="type", values="value")
    .reset_index()
    .rename_axis(columns=None)
)

for column in ["pm10", "pm25"]:
    if column not in hourly_pm.columns:
        hourly_pm[column] = pd.NA

hourly_pm = hourly_pm[["sensorId", "timestamp", "pm10", "pm25"]]

print(f"Hourly PM rows: {len(hourly_pm):,}")
print(f"Sensors with any PM data: {hourly_pm['sensorId'].nunique():,}")
hourly_pm.head()

Pulse files found: 2,418


/tmp/ipykernel_351347/1228433216.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pulse = pd.concat(pulse_parts, ignore_index=True)


Hourly PM rows: 147,379
Sensors with any PM data: 88


,sensorId,timestamp,pm10,pm25
0,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-12-01 00:00:00+00:00,10.50,6.00
1,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-12-01 01:00:00+00:00,9.00,4.50
2,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-12-01 02:00:00+00:00,14.00,6.00
3,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-12-01 03:00:00+00:00,12.75,7.25
4,007f2b03-94e6-47b3-9e3e-44273354acd5,2025-12-01 04:00:00+00:00,14.25,7.25


## 3. Join PM onto the hourly weather grid

This keeps all weather-grid rows. Missing PM stays missing.

In [ ]:
merged = weather.merge(
    hourly_pm,
    on=["sensorId", "timestamp"],
    how="left",
)

missing_by_sensor = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.isna().mean() * 100)
    .round(2)
    .sort_values("pm10", ascending=False)
)

has_pm_data = (
    merged
    .groupby("sensorId")[["pm10", "pm25"]]
    .apply(lambda frame: frame.notna().any())
)

print("Sensors with any PM data:")
print(has_pm_data.sum())

missing_by_sensor_100 = missing_by_sensor[(missing_by_sensor["pm10"] == 100) & (missing_by_sensor["pm25"] == 100)]
print(f"Sensors with 100% missing PM values: {len(missing_by_sensor_100):,}")

missing_by_sensor

Sensors with any PM data:
pm10    88
pm25    87
dtype: int64
Sensors with 100% missing PM values: 98


,pm10,pm25
sensorId,,
c364c259-c01c-4ae7-b1d3-dde42e56c196,100.00,100.00
sensor_dev_39544_399,100.00,100.00
e7a05c01-1d5c-479a-a5a5-419f28cebeef,100.00,100.00
ef8fbcf0-e04e-4d15-ab3c-a625a2f9245d,100.00,100.00
f1b3d48c-c8ae-411e-9bd9-6ab057600709,100.00,100.00
...,...,...
sensor_dev_80195_553,0.46,0.46
35bdd494-5395-4ac5-b7b9-05b82c9b6acd,0.46,0.46
8a855889-8853-4777-a0e4-107cf78ab550,0.46,0.46


## 4. Optionally drop sensors with too much missing PM data

By default this keeps only sensors where both PM10 and PM2.5 missingness is <= 50%.

Unlike the previous observed-only CSV, this does not drop individual missing rows. It keeps the hourly rows and leaves PM as NaN.

In [6]:
if MAX_MISSING_PERCENT is None:
    kept_sensors = missing_by_sensor.index
else:
    kept_sensors = missing_by_sensor[
        (missing_by_sensor["pm10"] <= MAX_MISSING_PERCENT)
        & (missing_by_sensor["pm25"] <= MAX_MISSING_PERCENT)
    ].index

hourly_nan = (
    merged[merged["sensorId"].isin(kept_sensors)]
    .sort_values(["timestamp", "sensorId"])
    .reset_index(drop=True)
)

print(f"Kept sensors: {len(kept_sensors):,} / {missing_by_sensor.shape[0]:,}")
print(f"Output rows: {len(hourly_nan):,}")
print("Missing PM percent in output:")
print((hourly_nan[["pm10", "pm25"]].isna().mean() * 100).round(2))

hourly_nan.head()

Kept sensors: 73 / 186
Output rows: 159,432
Missing PM percent in output:
pm10    12.96
pm25    12.96
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-11-30 22:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,2.5000,94.795290,2.817445,26.564985,947.18340,NaN,NaN
1,2025-11-30 22:00:00+00:00,01440b05-255d-4764-be87-bdf135f32289,42.027619,21.387420,4.5795,88.728820,0.763675,315.000100,985.35610,NaN,NaN
2,2025-11-30 22:00:00+00:00,01cf1cec-bf2d-41b3-8cd5-e8bd720f01b4,41.993499,21.445131,5.1000,86.912544,0.648999,123.690094,987.87190,NaN,NaN
3,2025-11-30 22:00:00+00:00,089aa02d-203e-4462-a395-b7dd86692480,41.997294,21.424585,5.1000,86.912544,0.648999,123.690094,987.14874,NaN,NaN
4,2025-11-30 22:00:00+00:00,0a058579-12c9-47be-971b-607198002d3b,41.993972,21.426850,5.1000,86.912544,0.648999,123.690094,987.02810,NaN,NaN


## 5. Optionally interpolate short PM gaps

When enabled, this fills only short consecutive missing PM gaps per sensor. Long outages remain NaN.

In [7]:
def missing_pm_summary(df):
    return pd.DataFrame(
        {
            "missing_count": df[["pm10", "pm25"]].isna().sum(),
            "missing_percent": (df[["pm10", "pm25"]].isna().mean() * 100).round(2),
        }
    )


def missing_pm_by_sensor(df):
    return (
        df
        .groupby("sensorId")[["pm10", "pm25"]]
        .apply(lambda frame: frame.isna().mean() * 100)
        .round(2)
        .sort_values("pm10", ascending=False)
    )


def interpolate_short_pm_gaps(df, value_columns=("pm10", "pm25"), max_gap_hours=6):
    result = df.sort_values(["sensorId", "timestamp"]).copy()

    for column in value_columns:
        result[column] = (
            result
            .groupby("sensorId", group_keys=False)[column]
            .apply(
                lambda series: series.interpolate(
                    method="linear",
                    limit=max_gap_hours,
                    limit_area="inside",
                )
            )
        )

    return result

missing_before_interpolation = missing_pm_summary(hourly_nan)
missing_by_sensor_before_interpolation = missing_pm_by_sensor(hourly_nan)

if INTERPOLATE_SHORT_GAPS:
    hourly_nan = interpolate_short_pm_gaps(
        hourly_nan,
        max_gap_hours=MAX_INTERPOLATION_GAP_HOURS,
    )
    print(f"Interpolated PM gaps up to {MAX_INTERPOLATION_GAP_HOURS} consecutive hours.")
else:
    print("Short-gap interpolation disabled. Missing PM values remain NaN.")

missing_after_interpolation = missing_pm_summary(hourly_nan)
missing_by_sensor_after_interpolation = missing_pm_by_sensor(hourly_nan)

missing_change = missing_before_interpolation.join(
    missing_after_interpolation,
    lsuffix="_before",
    rsuffix="_after",
)
missing_change["filled_count"] = (
    missing_change["missing_count_before"] - missing_change["missing_count_after"]
)
missing_change["filled_percent_points"] = (
    missing_change["missing_percent_before"] - missing_change["missing_percent_after"]
).round(2)

missing_by_sensor_change = missing_by_sensor_before_interpolation.join(
    missing_by_sensor_after_interpolation,
    lsuffix="_before",
    rsuffix="_after",
)
missing_by_sensor_change["pm10_filled_percent_points"] = (
    missing_by_sensor_change["pm10_before"] - missing_by_sensor_change["pm10_after"]
).round(2)
missing_by_sensor_change["pm25_filled_percent_points"] = (
    missing_by_sensor_change["pm25_before"] - missing_by_sensor_change["pm25_after"]
).round(2)

print("Overall missingness before/after interpolation:")
display(missing_change)

print("Missingness by sensor before/after interpolation:")
missing_by_sensor_change

Interpolated PM gaps up to 6 consecutive hours.
Overall missingness before/after interpolation:


,missing_count_before,missing_percent_before,missing_count_after,missing_percent_after,filled_count,filled_percent_points
pm10,20663,12.96,17229,10.81,3434,2.15
pm25,20663,12.96,17229,10.81,3434,2.15


Missingness by sensor before/after interpolation:


,pm10_before,pm25_before,pm10_after,pm25_after,pm10_filled_percent_points,pm25_filled_percent_points
sensorId,,,,,,
a52857b4-3d12-4520-92e5-b54f1e367ca6,49.18,49.18,49.18,49.18,0.00,0.00
1000,43.54,43.45,39.79,39.79,3.75,3.66
1001,42.08,41.94,38.78,38.78,3.30,3.16
3568aa20-235a-408c-861b-279c9f4d7709,41.90,41.90,41.90,41.90,0.00,0.00
1003,41.85,41.76,38.78,38.78,3.07,2.98
...,...,...,...,...,...,...
sensor_dev_78844_374,0.46,0.46,0.18,0.18,0.28,0.28
sensor_dev_62788_603,0.46,0.46,0.18,0.18,0.28,0.28
a880569d-4dcc-467c-8e51-2610d272ff9c,0.46,0.46,0.18,0.18,0.28,0.28


In [8]:
# filter to show only nan values for pm10 and pm25
hourly_nan[hourly_nan[["pm10", "pm25"]].isna().any(axis=1)]


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-11-30 22:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,2.50,94.795290,2.817445,26.564985,947.18340,NaN,NaN
73,2025-11-30 23:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.00,95.766060,2.340000,22.619910,947.08930,NaN,NaN
18177,2025-12-11 07:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.65,79.588280,3.081104,353.290250,952.64460,NaN,NaN
18250,2025-12-11 08:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,5.80,65.355934,3.319036,347.471200,953.38980,NaN,NaN
50881,2025-12-29 23:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-3.75,66.146450,2.952219,37.568665,943.65295,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
159139,2026-03-01 17:00:00+00:00,sensor_dev_82703_642,42.000000,21.464000,6.10,73.148026,3.065289,310.236300,996.25085,NaN,NaN
159212,2026-03-01 18:00:00+00:00,sensor_dev_82703_642,42.000000,21.464000,8.20,64.960380,1.808978,5.710507,996.08880,NaN,NaN
159285,2026-03-01 19:00:00+00:00,sensor_dev_82703_642,42.000000,21.464000,7.30,68.331150,4.510787,298.610350,996.28326,NaN,NaN
159358,2026-03-01 20:00:00+00:00,sensor_dev_82703_642,42.000000,21.464000,6.90,64.898544,2.421652,311.987120,996.53140,NaN,NaN


## 6. Save CSV

Missing PM values are written as blank cells. When you load this CSV with pandas later, those blanks become NaN.

In [9]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
hourly_nan.to_csv(OUTPUT_CSV, index=False, na_rep="")

print(f"Saved: {OUTPUT_CSV}")

Saved: /home/nikola/realtime-air-quality-forecasting/skopje/data/streaming/skopje_sensor_weather_features_online_short_gap_interpolated.csv


## 7. Verify that blanks reload as NaN

In [10]:
check = pd.read_csv(OUTPUT_CSV)

print(f"Reloaded rows: {len(check):,}")
print("Missing values after reloading:")
print(check[["pm10", "pm25"]].isna().sum())
print("Missing percent after reloading:")
print((check[["pm10", "pm25"]].isna().mean() * 100).round(2))

check[check[["pm10", "pm25"]].isna().any(axis=1)].head(20)

Reloaded rows: 159,432
Missing values after reloading:
pm10    17229
pm25    17229
dtype: int64
Missing percent after reloading:
pm10    10.81
pm25    10.81
dtype: float64


,timestamp,sensorId,lat,lon,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,pm10,pm25
0,2025-11-30 22:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,2.50,94.795290,2.817445,26.564985,947.18340,NaN,NaN
1,2025-11-30 23:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.00,95.766060,2.340000,22.619910,947.08930,NaN,NaN
249,2025-12-11 07:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,1.65,79.588280,3.081104,353.290250,952.64460,NaN,NaN
250,2025-12-11 08:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,5.80,65.355934,3.319036,347.471200,953.38980,NaN,NaN
697,2025-12-29 23:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-3.75,66.146450,2.952219,37.568665,943.65295,NaN,NaN
698,2025-12-30 00:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-4.15,72.268290,1.405845,50.194473,943.64276,NaN,NaN
699,2025-12-30 01:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-5.00,71.816990,2.545584,44.999897,943.14355,NaN,NaN
700,2025-12-30 02:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-5.15,72.352264,3.184525,47.290634,942.45496,NaN,NaN
701,2025-12-30 03:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-4.95,71.827070,3.563818,44.999897,941.67130,NaN,NaN
702,2025-12-30 04:00:00+00:00,007f2b03-94e6-47b3-9e3e-44273354acd5,42.055623,21.305011,-4.60,68.596300,4.072935,44.999897,940.83405,NaN,NaN
